In [1]:
print("Hello! Jupyter is working.")

Hello! Jupyter is working.


In [2]:
import pandas as pd
import os

print("Files in dataset folder:")
print(os.listdir("../dataset"))

Files in dataset folder:
['job_roles.csv', 'skills_database.json', 'skills_list.csv', 'test_resumes.json', 'training_data.csv']


In [3]:
df = pd.read_csv("../dataset/training_data.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()

Dataset shape: (10000, 7)

Columns:
['Resume ID', 'Resume Text', 'Education', 'Experience Years', 'Skills', 'Job Role', 'Category']


,Resume ID,Resume Text,Education,Experience Years,Skills,Job Role,Category
0,R000000,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,2,Market Research|Maven|Java|REST API|Spring Boo...,Java Backend Developer,Technology
1,R000001,Education: Master's in Microbiology Experience...,Master's in Microbiology,3,Agile|Data Analysis|Precision|Problem Solving|...,Microbiologist,Science & Research
2,R000002,Education: Apprenticeship Experience: 2 years ...,Apprenticeship,2,React|Customer Service|Technical Knowledge|Plu...,Plumber,Skilled Trades
3,R000003,Education: Bachelor's in Computer Science Expe...,Bachelor's in Computer Science,0,MySQL|Query Optimization|Symfony|Terraform|PHP...,PHP Developer,Technology
4,R000004,Education: Bachelor's in Design Experience: 3 ...,Bachelor's in Design,3,Problem Solving|Docker|Creativity|Scrum|Ansibl...,Art Director,Creative & Design


In [4]:
print("Missing values:")
print(df.isnull().sum())

print("\nNumber of unique job roles:")
print(df["Job Role"].nunique())

print("\nJob roles:")
print(df["Job Role"].unique())

print("\nCategory distribution:")
print(df["Category"].value_counts())

Missing values:
Resume ID           0
Resume Text         0
Education           0
Experience Years    0
Skills              0
Job Role            0
Category            0
dtype: int64

Number of unique job roles:
324

Job roles:
<ArrowStringArray>
['Java Backend Developer',         'Microbiologist',                'Plumber',
          'PHP Developer',           'Art Director',    'Executive Assistant',
    'Cassandra Developer',  'Informatica Developer',     'Physical Therapist',
       'Elixir Developer',
 ...
         'Tax Specialist',      'Event Coordinator',         'SEO Specialist',
    'Procurement Manager',  'Management Consultant',   'Social Media Manager',
            'News Anchor',             'Tour Guide',                'Athlete',
           'Ship Captain']
Length: 324, dtype: str

Category distribution:
Category
Technology                      2511
Data & Analytics                 568
Healthcare                       488
Marketing & Sales                463
Engineering & M

In [5]:
role_counts = df["Job Role"].value_counts()

print("Total job roles:", len(role_counts))
print("\nMost common job roles:")
print(role_counts.head(15))

print("\nLeast common job roles:")
print(role_counts.tail(15))

print("\nStatistics:")
print(role_counts.describe())

Total job roles: 324

Most common job roles:
Job Role
Java Backend Developer    50
Automotive Engineer       47
Electronics Engineer      44
School Principal          44
Announcer                 43
Hairstylist               43
Cashier                   42
Audit Manager             42
Executive Assistant       41
HVAC Technician           41
Medical Technologist      41
Grant Writer              41
Microbiologist            40
Conservation Officer      40
Museum Curator            40
Name: count, dtype: int64

Least common job roles:
Job Role
Project Manager              22
Procurement Manager          22
Athlete                      22
Strategy Consultant          21
Psychologist                 21
Animator                     21
SOC Analyst                  20
CFO                          20
Quality Assurance Manager    20
Occupational Therapist       20
Personal Trainer             20
Controller                   20
Oracle DBA                   19
Lawyer                       18
Car

In [6]:
df["ML_Text"] = (
    df["Resume Text"].astype(str) + " " +
    df["Skills"].astype(str)
)

print(df[["Resume Text", "Skills", "ML_Text"]].head())

                                         Resume Text  \
0  Education: Bachelor's in Computer Science Expe...   
1  Education: Master's in Microbiology Experience...   
2  Education: Apprenticeship Experience: 2 years ...   
3  Education: Bachelor's in Computer Science Expe...   
4  Education: Bachelor's in Design Experience: 3 ...   

                                              Skills  \
0  Market Research|Maven|Java|REST API|Spring Boo...   
1  Agile|Data Analysis|Precision|Problem Solving|...   
2  React|Customer Service|Technical Knowledge|Plu...   
3  MySQL|Query Optimization|Symfony|Terraform|PHP...   
4  Problem Solving|Docker|Creativity|Scrum|Ansibl...   

                                             ML_Text  
0  Education: Bachelor's in Computer Science Expe...  
1  Education: Master's in Microbiology Experience...  
2  Education: Apprenticeship Experience: 2 years ...  
3  Education: Bachelor's in Computer Science Expe...  
4  Education: Bachelor's in Design Experience: 3 ..

In [7]:
print("Example ML text:\n")
print(df["ML_Text"].iloc[0])

print("\nText length:")
print(df["ML_Text"].str.len().describe())

Example ML text:

Education: Bachelor's in Computer Science Experience: 2 years Skills: Market Research, Maven, Java, REST API, Spring Boot, Microservices, Statistics, SQL Market Research|Maven|Java|REST API|Spring Boot|Microservices|Statistics|SQL

Text length:
count    10000.000000
mean       287.661200
std         33.692066
min        172.000000
25%        265.000000
50%        289.000000
75%        311.000000
max        421.000000
Name: ML_Text, dtype: float64


In [8]:
from sklearn.model_selection import train_test_split

X = df["ML_Text"]
y = df["Job Role"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training job roles:", y_train.nunique())
print("Testing job roles:", y_test.nunique())

Training samples: 8000
Testing samples: 2000
Training job roles: 324
Testing job roles: 324


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Testing TF-IDF shape:", X_test_tfidf.shape)
print("Number of features:", len(tfidf.get_feature_names_out()))

Training TF-IDF shape: (8000, 5000)
Testing TF-IDF shape: (2000, 5000)
Number of features: 5000


In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [11]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test_tfidf)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print("\nAccuracy percentage:", round(accuracy * 100, 2), "%")

Accuracy: 0.9995

Accuracy percentage: 99.95 %


In [12]:
print(classification_report(y_test, y_pred, zero_division=0))

                                 precision    recall  f1-score   support

                      3D Artist       1.00      1.00      1.00         7
               AI/ML Specialist       1.00      1.00      1.00         5
                AR/VR Developer       1.00      1.00      1.00         7
               Academic Advisor       1.00      1.00      1.00         5
          Academic Data Analyst       1.00      1.00      1.00         6
              Account Executive       1.00      1.00      1.00         7
                     Accountant       1.00      1.00      1.00         7
       Administrative Assistant       1.00      1.00      1.00         6
             Aerospace Engineer       1.00      1.00      1.00         8
          Agricultural Engineer       1.00      1.00      1.00         7
             Analytics Engineer       1.00      1.00      1.00         6
              Angular Developer       1.00      1.00      1.00         6
                       Animator       1.00      1.

In [13]:
from sklearn.metrics import confusion_matrix
import numpy as np

cm = confusion_matrix(y_test, y_pred)

print("Total test samples:", len(y_test))
print("Correct predictions:", np.trace(cm))
print("Incorrect predictions:", len(y_test) - np.trace(cm))

Total test samples: 2000
Correct predictions: 1999
Incorrect predictions: 1


In [14]:
errors = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

errors = errors[errors["Actual"] != errors["Predicted"]]

print("Number of errors:", len(errors))
print(errors)

Number of errors: 1
         Actual  Predicted
1764  Head Chef  Sous Chef


In [15]:
# Check whether the actual job role appears directly in ML_Text

def role_in_text(row):
    role = str(row["Job Role"]).lower()
    text = str(row["ML_Text"]).lower()
    return role in text

df["Role_In_Text"] = df.apply(role_in_text, axis=1)

print("Resumes containing their exact Job Role:")
print(df["Role_In_Text"].value_counts())

print("\nPercentage containing job role:")
print(round(df["Role_In_Text"].mean() * 100, 2), "%")

Resumes containing their exact Job Role:
Role_In_Text
False    9417
True      583
Name: count, dtype: int64

Percentage containing job role:
5.83 %


In [16]:
wrong_index = errors.index[0]

print("ACTUAL ROLE:", df.loc[wrong_index, "Job Role"])
print("PREDICTED ROLE:", errors.iloc[0]["Predicted"])

print("\nRESUME TEXT:")
print(df.loc[wrong_index, "Resume Text"])

print("\nSKILLS:")
print(df.loc[wrong_index, "Skills"])

print("\nEDUCATION:")
print(df.loc[wrong_index, "Education"])

print("\nEXPERIENCE:")
print(df.loc[wrong_index, "Experience Years"])

ACTUAL ROLE: Data Scientist
PREDICTED ROLE: Sous Chef

RESUME TEXT:
Education: Master's in Data Science Experience: 4 years Skills: TensorFlow, Data Analysis, Statistics, Machine Learning, React, Mobile UI/UX, Python, SQL

SKILLS:
TensorFlow|Data Analysis|Statistics|Machine Learning|React|Mobile UI/UX|Python|SQL

EDUCATION:
Master's in Data Science

EXPERIENCE:
4


In [17]:
# MODEL A: Resume Text only

X_text = df["Resume Text"]
y = df["Job Role"]

X_text_train, X_text_test, y_text_train, y_text_test = train_test_split(
    X_text,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

tfidf_text = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_text_train_tfidf = tfidf_text.fit_transform(X_text_train)
X_text_test_tfidf = tfidf_text.transform(X_text_test)

model_text = LogisticRegression(
    max_iter=2000,
    random_state=42
)

model_text.fit(X_text_train_tfidf, y_text_train)

y_text_pred = model_text.predict(X_text_test_tfidf)

text_accuracy = accuracy_score(y_text_test, y_text_pred)

print("Resume Text Only Accuracy:",
      round(text_accuracy * 100, 2), "%")

Resume Text Only Accuracy: 99.2 %


In [18]:
# MODEL B: Skills only

X_skills = df["Skills"]

X_skills_train, X_skills_test, y_skills_train, y_skills_test = train_test_split(
    X_skills,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

tfidf_skills = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_skills_train_tfidf = tfidf_skills.fit_transform(X_skills_train)
X_skills_test_tfidf = tfidf_skills.transform(X_skills_test)

model_skills = LogisticRegression(
    max_iter=2000,
    random_state=42
)

model_skills.fit(X_skills_train_tfidf, y_skills_train)

y_skills_pred = model_skills.predict(X_skills_test_tfidf)

skills_accuracy = accuracy_score(y_skills_test, y_skills_pred)

print("Skills Only Accuracy:",
      round(skills_accuracy * 100, 2), "%")

Skills Only Accuracy: 100.0 %


In [19]:
# Check whether identical skill sets are associated with multiple job roles

skill_role_counts = (
    df.groupby("Skills")["Job Role"]
      .nunique()
      .sort_values(ascending=False)
)

print("Number of unique skill combinations:", len(skill_role_counts))

print("\nSkill combinations linked to multiple job roles:")
print(skill_role_counts[skill_role_counts > 1].head(20))

print("\nSkill combinations linked to exactly one job role:")
print((skill_role_counts == 1).sum())

Number of unique skill combinations: 10000

Skill combinations linked to multiple job roles:
Series([], Name: Job Role, dtype: int64)

Skill combinations linked to exactly one job role:
10000


In [20]:
print("Duplicate Resume Text:", df["Resume Text"].duplicated().sum())
print("Duplicate Skills:", df["Skills"].duplicated().sum())
print("Duplicate ML_Text:", df["ML_Text"].duplicated().sum())

Duplicate Resume Text: 0
Duplicate Skills: 0
Duplicate ML_Text: 0


In [21]:
import numpy as np

# Shuffle the training labels randomly
y_train_random = y_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# Reset X_train index so they align correctly
X_train_reset = X_train.reset_index(drop=True)

# Train TF-IDF on the same training text
tfidf_random = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    stop_words="english"
)

X_train_random_tfidf = tfidf_random.fit_transform(X_train_reset)
X_test_random_tfidf = tfidf_random.transform(X_test.reset_index(drop=True))

# Train model with random labels
random_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

random_model.fit(X_train_random_tfidf, y_train_random)

# Predict
random_pred = random_model.predict(X_test_random_tfidf)

random_accuracy = accuracy_score(y_test.reset_index(drop=True), random_pred)

print("Random-label accuracy:",
      round(random_accuracy * 100, 2), "%")

Random-label accuracy: 0.2 %


In [22]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

svm_model = LinearSVC(
    random_state=42
)

svm_model.fit(X_train_tfidf, y_train)

y_svm_pred = svm_model.predict(X_test_tfidf)

svm_accuracy = accuracy_score(y_test, y_svm_pred)
svm_f1 = f1_score(y_test, y_svm_pred, average="weighted")

print("Linear SVM Accuracy:",
      round(svm_accuracy * 100, 2), "%")

print("Linear SVM Weighted F1:",
      round(svm_f1 * 100, 2), "%")

Linear SVM Accuracy: 100.0 %
Linear SVM Weighted F1: 100.0 %


In [23]:
import joblib

# Save the trained Linear SVM model
joblib.dump(svm_model, "../models/job_role_model.pkl")

# Save the TF-IDF vectorizer
joblib.dump(tfidf, "../models/tfidf_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [24]:
# Load the saved model and vectorizer

loaded_model = joblib.load("../models/job_role_model.pkl")
loaded_tfidf = joblib.load("../models/tfidf_vectorizer.pkl")

print("Saved model loaded successfully!")

Saved model loaded successfully!


In [25]:
new_resume = """
Bachelor's degree in Computer Science.
Experienced in Python, Pandas, NumPy, Scikit-learn and Machine Learning.
Worked on data analysis, predictive modeling and visualization.
Skills include Python, SQL, Machine Learning, Statistics and TensorFlow.
"""

new_resume_tfidf = loaded_tfidf.transform([new_resume])

prediction = loaded_model.predict(new_resume_tfidf)

print("Predicted Job Role:", prediction[0])

Predicted Job Role: Data Scientist


In [26]:
# Inspect skills_list.csv

skills_df = pd.read_csv("../dataset/skills_list.csv")

print("Shape:", skills_df.shape)
print("\nColumns:")
print(skills_df.columns.tolist())

print("\nFirst 10 rows:")
print(skills_df.head(10))

Shape: (120, 2)

Columns:
['Skill Name', 'Category']

First 10 rows:
   Skill Name     Category
0      Python  Programming
1        Java  Programming
2  JavaScript  Programming
3         C++  Programming
4          C#  Programming
5        Ruby  Programming
6         PHP  Programming
7          Go  Programming
8        Rust  Programming
9  TypeScript  Programming


In [27]:
import json

with open("../dataset/skills_database.json", "r", encoding="utf-8") as f:
    skills_data = json.load(f)

print("JSON data type:", type(skills_data))

if isinstance(skills_data, dict):
    print("\nJSON keys:")
    print(list(skills_data.keys())[:20])
    
elif isinstance(skills_data, list):
    print("\nNumber of items:", len(skills_data))
    print("\nFirst item:")
    print(skills_data[0])

JSON data type: <class 'dict'>

JSON keys:
['Programming', 'Web Development', 'Mobile Development', 'Cloud & DevOps', 'Data Science & Analytics', 'Database', 'Soft Skills', 'Business', 'Design', 'Cybersecurity', 'Professional']


In [28]:
# Create our skill vocabulary

skill_list = skills_df["Skill Name"].dropna().astype(str).tolist()

print("Total skills:", len(skill_list))
print("\nFirst 20 skills:")
print(skill_list[:20])

Total skills: 120

First 20 skills:
['Python', 'Java', 'JavaScript', 'C++', 'C#', 'Ruby', 'PHP', 'Go', 'Rust', 'TypeScript', 'Kotlin', 'Swift', 'Scala', 'Perl', 'Shell Scripting', 'R', 'HTML', 'CSS', 'React', 'Vue.js']


In [29]:
import re

def extract_skills(text, skill_list):
    text = str(text).lower()
    found_skills = []

    for skill in skill_list:
        skill_lower = skill.lower()

        # Match the skill as a whole word/phrase
        pattern = r"(?<!\w)" + re.escape(skill_lower) + r"(?!\w)"

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

In [30]:
test_resume = """
I am a Data Scientist with experience in Python, SQL and Machine Learning.
I have worked with Pandas, TensorFlow and JavaScript.
I also have experience with AWS and Docker.
"""

detected_skills = extract_skills(test_resume, skill_list)

print("Detected Skills:")
for skill in detected_skills:
    print("✓", skill)

Detected Skills:
✓ Python
✓ JavaScript
✓ AWS
✓ Docker
✓ Machine Learning
✓ TensorFlow
✓ Pandas
✓ SQL
✓ Python
✓ SQL


In [31]:
# Check for duplicate skill names

duplicate_skills = skills_df[
    skills_df["Skill Name"].duplicated(keep=False)
]

print("Duplicate skill entries:")
print(duplicate_skills)

Duplicate skill entries:
             Skill Name                  Category
0                Python               Programming
15                    R               Programming
57                  SQL  Data Science & Analytics
61               Python  Data Science & Analytics
62                    R  Data Science & Analytics
63                  SQL                  Database
83   Project Management               Soft Skills
84   Strategic Planning               Soft Skills
89   Strategic Planning                  Business
111  Project Management              Professional


In [32]:
skill_list = (
    skills_df["Skill Name"]
    .dropna()
    .astype(str)
    .drop_duplicates()
    .tolist()
)

print("Original skill count:", len(skills_df))
print("Unique skill count:", len(skill_list))

Original skill count: 120
Unique skill count: 115


In [33]:
detected_skills = extract_skills(test_resume, skill_list)

print("Detected Skills:")
for skill in detected_skills:
    print("✓", skill)

Detected Skills:
✓ Python
✓ JavaScript
✓ AWS
✓ Docker
✓ Machine Learning
✓ TensorFlow
✓ Pandas
✓ SQL


In [34]:
def analyze_resume(resume_text):
    
    # 1. Extract skills
    detected_skills = extract_skills(resume_text, skill_list)
    
    # 2. Create ML input
    skills_text = " ".join(detected_skills)
    ml_text = str(resume_text) + " " + skills_text
    
    # 3. Convert text to TF-IDF
    resume_tfidf = loaded_tfidf.transform([ml_text])
    
    # 4. Predict job role
    predicted_role = loaded_model.predict(resume_tfidf)[0]
    
    # 5. Return analysis
    return {
        "predicted_role": predicted_role,
        "detected_skills": detected_skills
    }

In [35]:
resume = """
I am a Data Scientist with 4 years of experience.

I have a Master's degree in Data Science.
My technical skills include Python, SQL, Pandas, NumPy,
Machine Learning, TensorFlow and Statistics.

I have worked on predictive modeling, data analysis
and machine learning projects.
"""

result = analyze_resume(resume)

print("PREDICTED JOB ROLE:")
print(result["predicted_role"])

print("\nDETECTED SKILLS:")
for skill in result["detected_skills"]:
    print("✓", skill)

PREDICTED JOB ROLE:
Data Scientist

DETECTED SKILLS:
✓ Python
✓ Machine Learning
✓ TensorFlow
✓ Data Analysis
✓ Statistics
✓ Pandas
✓ NumPy
✓ SQL


In [36]:
# Inspect job roles dataset

job_roles_df = pd.read_csv("../dataset/job_roles.csv")

print("Shape:", job_roles_df.shape)

print("\nColumns:")
print(job_roles_df.columns.tolist())

print("\nFirst 10 rows:")
print(job_roles_df.head(10))

Shape: (324, 6)

Columns:
['Job Title', 'Category', 'Education Requirement', 'Experience Years', 'Required Skills', 'Salary Range']

First 10 rows:
                   Job Title    Category  \
0          Software Engineer  Technology   
1       Full Stack Developer  Technology   
2         Frontend Developer  Technology   
3          Backend Developer  Technology   
4            DevOps Engineer  Technology   
5            Cloud Architect  Technology   
6             Data Scientist  Technology   
7  Machine Learning Engineer  Technology   
8              Data Engineer  Technology   
9     Database Administrator  Technology   

                               Education Requirement  Experience Years  \
0  Bachelor's in Computer Science|Bachelor's in E...                 2   
1       Diploma in IT|Bachelor's in Computer Science                 2   
2       Diploma in IT|Bachelor's in Computer Science                 1   
3                     Bachelor's in Computer Science                 2 

In [37]:
def skill_gap_analysis(predicted_role, detected_skills):
    
    # Find the job role in job_roles.csv
    role_data = job_roles_df[
        job_roles_df["Job Title"].str.lower() == predicted_role.lower()
    ]
    
    if role_data.empty:
        return {
            "required_skills": [],
            "matched_skills": [],
            "missing_skills": []
        }
    
    # Get required skills
    required_skills_text = role_data.iloc[0]["Required Skills"]
    
    required_skills = [
        skill.strip()
        for skill in str(required_skills_text).split("|")
        if skill.strip()
    ]
    
    # Normalize detected skills for comparison
    detected_lower = {
        skill.lower().strip()
        for skill in detected_skills
    }
    
    # Find matched and missing skills
    matched_skills = []
    missing_skills = []
    
    for skill in required_skills:
        if skill.lower().strip() in detected_lower:
            matched_skills.append(skill)
        else:
            missing_skills.append(skill)
    
    return {
        "required_skills": required_skills,
        "matched_skills": matched_skills,
        "missing_skills": missing_skills
    }

In [38]:
gap_result = skill_gap_analysis(
    result["predicted_role"],
    result["detected_skills"]
)

print("PREDICTED ROLE:")
print(result["predicted_role"])

print("\nREQUIRED SKILLS:")
for skill in gap_result["required_skills"]:
    print("•", skill)

print("\nMATCHED SKILLS:")
for skill in gap_result["matched_skills"]:
    print("✓", skill)

print("\nMISSING SKILLS:")
for skill in gap_result["missing_skills"]:
    print("✗", skill)

PREDICTED ROLE:
Data Scientist

REQUIRED SKILLS:
• Python
• Machine Learning
• Statistics
• TensorFlow
• SQL
• Data Analysis

MATCHED SKILLS:
✓ Python
✓ Machine Learning
✓ Statistics
✓ TensorFlow
✓ SQL
✓ Data Analysis

MISSING SKILLS:


In [39]:
required_count = len(gap_result["required_skills"])
matched_count = len(gap_result["matched_skills"])

if required_count > 0:
    skill_match_percentage = (
        matched_count / required_count
    ) * 100
else:
    skill_match_percentage = 0

print("Skill Match Score:",
      round(skill_match_percentage, 2), "%")

Skill Match Score: 100.0 %


In [40]:
import re

def extract_experience_years(resume_text):
    
    text = str(resume_text).lower()
    
    patterns = [
        r'(\d+)\+?\s*years?\s*(?:of\s*)?(?:experience|work experience)',
        r'(\d+)\+?\s*years?\s*experience'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        
        if match:
            return int(match.group(1))
    
    return 0

In [41]:
candidate_experience = extract_experience_years(resume)

print("Candidate Experience:", candidate_experience, "years")

Candidate Experience: 4 years


In [42]:
def get_role_requirements(predicted_role):
    
    role_data = job_roles_df[
        job_roles_df["Job Title"].str.lower() == predicted_role.lower()
    ]
    
    if role_data.empty:
        return None
    
    row = role_data.iloc[0]
    
    return {
        "education": str(row["Education Requirement"]),
        "experience": int(row["Experience Years"]),
        "salary": str(row["Salary Range"]),
        "category": str(row["Category"])
    }

In [43]:
role_requirements = get_role_requirements(
    result["predicted_role"]
)

print("ROLE:", result["predicted_role"])
print("CATEGORY:", role_requirements["category"])
print("EDUCATION:", role_requirements["education"])
print("EXPERIENCE REQUIRED:", role_requirements["experience"], "years")
print("SALARY RANGE:", role_requirements["salary"])

ROLE: Data Scientist
CATEGORY: Technology
EDUCATION: Master's in Data Science|Bachelor's in Mathematics
EXPERIENCE REQUIRED: 2 years
SALARY RANGE: 90-160K


In [44]:
required_experience = role_requirements["experience"]

if candidate_experience >= required_experience:
    experience_status = "Meets requirement"
else:
    experience_status = "Below requirement"

print("Candidate Experience:",
      candidate_experience, "years")

print("Required Experience:",
      required_experience, "years")

print("Experience Status:",
      experience_status)

Candidate Experience: 4 years
Required Experience: 2 years
Experience Status: Meets requirement


In [45]:
def check_education(resume_text, education_requirement):
    
    resume_lower = str(resume_text).lower()
    
    education_options = [
        edu.strip()
        for edu in str(education_requirement).split("|")
        if edu.strip()
    ]
    
    matched_education = []
    
    for education in education_options:
        
        # Compare important words rather than requiring exact sentences
        education_words = re.findall(
            r'\b[a-zA-Z]+\b',
            education.lower()
        )
        
        important_words = [
            word for word in education_words
            if word not in {
                "in", "and", "of", "or"
            }
        ]
        
        if important_words:
            matches = sum(
                word in resume_lower
                for word in important_words
            )
            
            if matches >= max(1, len(important_words) * 0.6):
                matched_education.append(education)
    
    return matched_education

In [46]:
matched_education = check_education(
    resume,
    role_requirements["education"]
)

print("Education Requirements:")
for edu in role_requirements["education"].split("|"):
    print("•", edu)

print("\nMatched Education:")
for edu in matched_education:
    print("✓", edu)

Education Requirements:
• Master's in Data Science
• Bachelor's in Mathematics

Matched Education:
✓ Master's in Data Science


In [47]:
def calculate_resume_score(
    skill_match_percentage,
    education_matched,
    candidate_experience,
    required_experience
):
    
    # Skill component: 60%
    skill_score = skill_match_percentage * 0.60
    
    # Education component: 20%
    education_score = 20 if education_matched else 0
    
    # Experience component: 20%
    if required_experience <= 0:
        experience_score = 20
    else:
        experience_ratio = min(
            candidate_experience / required_experience,
            1
        )
        experience_score = experience_ratio * 20
    
    total_score = (
        skill_score +
        education_score +
        experience_score
    )
    
    return round(total_score, 2)

In [48]:
education_matched = len(matched_education) > 0

resume_score = calculate_resume_score(
    skill_match_percentage,
    education_matched,
    candidate_experience,
    required_experience
)

print("Resume Score:", resume_score, "/ 100")

Resume Score: 100.0 / 100


In [49]:
def complete_resume_analysis(resume_text):
    
    # -----------------------------
    # 1. Extract skills
    # -----------------------------
    detected_skills = extract_skills(
        resume_text,
        skill_list
    )
    
    # -----------------------------
    # 2. Prepare ML text
    # -----------------------------
    skills_text = " ".join(detected_skills)
    ml_text = str(resume_text) + " " + skills_text
    
    # -----------------------------
    # 3. Predict job role
    # -----------------------------
    resume_tfidf = loaded_tfidf.transform([ml_text])
    predicted_role = loaded_model.predict(resume_tfidf)[0]
    
    # -----------------------------
    # 4. Get role requirements
    # -----------------------------
    role_requirements = get_role_requirements(
        predicted_role
    )
    
    # -----------------------------
    # 5. Skill gap analysis
    # -----------------------------
    gap_result = skill_gap_analysis(
        predicted_role,
        detected_skills
    )
    
    required_count = len(
        gap_result["required_skills"]
    )
    
    matched_count = len(
        gap_result["matched_skills"]
    )
    
    if required_count > 0:
        skill_match = (
            matched_count / required_count
        ) * 100
    else:
        skill_match = 0
    
    # -----------------------------
    # 6. Experience
    # -----------------------------
    candidate_experience = extract_experience_years(
        resume_text
    )
    
    required_experience = role_requirements[
        "experience"
    ]
    
    experience_meets = (
        candidate_experience >= required_experience
    )
    
    # -----------------------------
    # 7. Education
    # -----------------------------
    matched_education = check_education(
        resume_text,
        role_requirements["education"]
    )
    
    education_meets = len(matched_education) > 0
    
    # -----------------------------
    # 8. Resume match score
    # -----------------------------
    score = calculate_resume_score(
        skill_match,
        education_meets,
        candidate_experience,
        required_experience
    )
    
    # -----------------------------
    # 9. Return complete result
    # -----------------------------
    return {
        "predicted_role": predicted_role,
        "detected_skills": detected_skills,
        "required_skills": gap_result["required_skills"],
        "matched_skills": gap_result["matched_skills"],
        "missing_skills": gap_result["missing_skills"],
        "skill_match_percentage": round(skill_match, 2),
        "candidate_experience": candidate_experience,
        "required_experience": required_experience,
        "experience_meets": experience_meets,
        "education_requirements": role_requirements["education"],
        "matched_education": matched_education,
        "education_meets": education_meets,
        "salary_range": role_requirements["salary"],
        "category": role_requirements["category"],
        "resume_score": score
    }

In [50]:
final_result = complete_resume_analysis(resume)

print("=" * 50)
print("        AI RESUME ANALYZER RESULT")
print("=" * 50)

print("\n🎯 PREDICTED ROLE:")
print(final_result["predicted_role"])

print("\n🛠 DETECTED SKILLS:")
for skill in final_result["detected_skills"]:
    print("✓", skill)

print("\n📊 SKILL MATCH:")
print(final_result["skill_match_percentage"], "%")

print("\n❌ MISSING SKILLS:")
if final_result["missing_skills"]:
    for skill in final_result["missing_skills"]:
        print("✗", skill)
else:
    print("None")

print("\n🎓 EDUCATION:")
print(
    "Meets requirement"
    if final_result["education_meets"]
    else "Does not meet requirement"
)

print("\n💼 EXPERIENCE:")
print(
    final_result["candidate_experience"],
    "years / Required:",
    final_result["required_experience"],
    "years"
)

print(
    "Status:",
    "Meets requirement"
    if final_result["experience_meets"]
    else "Below requirement"
)

print("\n💰 SALARY RANGE:")
print(final_result["salary_range"])

print("\n⭐ RESUME MATCH SCORE:")
print(final_result["resume_score"], "/ 100")

print("=" * 50)

        AI RESUME ANALYZER RESULT

🎯 PREDICTED ROLE:
Data Scientist

🛠 DETECTED SKILLS:
✓ Python
✓ Machine Learning
✓ TensorFlow
✓ Data Analysis
✓ Statistics
✓ Pandas
✓ NumPy
✓ SQL

📊 SKILL MATCH:
100.0 %

❌ MISSING SKILLS:
None

🎓 EDUCATION:
Meets requirement

💼 EXPERIENCE:
4 years / Required: 2 years
Status: Meets requirement

💰 SALARY RANGE:
90-160K

⭐ RESUME MATCH SCORE:
100.0 / 100


In [51]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_svm = CalibratedClassifierCV(
    svm_model,
    method="sigmoid",
    cv=3
)

calibrated_svm.fit(
    X_train_tfidf,
    y_train
)

print("Calibrated SVM trained successfully!")

Calibrated SVM trained successfully!


In [52]:
calibrated_pred = calibrated_svm.predict(
    X_test_tfidf
)

calibrated_accuracy = accuracy_score(
    y_test,
    calibrated_pred
)

calibrated_f1 = f1_score(
    y_test,
    calibrated_pred,
    average="weighted"
)

print(
    "Calibrated SVM Accuracy:",
    round(calibrated_accuracy * 100, 2),
    "%"
)

print(
    "Calibrated SVM Weighted F1:",
    round(calibrated_f1 * 100, 2),
    "%"
)

Calibrated SVM Accuracy: 99.95 %
Calibrated SVM Weighted F1: 99.95 %


In [53]:
import joblib

joblib.dump(
    calibrated_svm,
    "../models/calibrated_job_role_model.pkl"
)

print("Calibrated SVM saved successfully!")

Calibrated SVM saved successfully!


In [54]:
# Test top 3 calibrated predictions

sample_resume = """
I am a Computer Science student with skills in
Python, Java, SQL, Git, software development and
problem solving.
"""

sample_skills = extract_skills(
    sample_resume,
    skill_list
)

sample_ml_text = (
    sample_resume
    + " "
    + " ".join(sample_skills)
)

sample_tfidf = tfidf.transform(
    [sample_ml_text]
)

probabilities = calibrated_svm.predict_proba(
    sample_tfidf
)[0]

top_indices = probabilities.argsort()[-3:][::-1]

print("TOP 3 JOB ROLE MATCHES\n")

for rank, index in enumerate(
    top_indices,
    start=1
):

    role = calibrated_svm.classes_[index]
    probability = probabilities[index] * 100

    print(
        f"{rank}. {role} — "
        f"{probability:.2f}%"
    )

TOP 3 JOB ROLE MATCHES

1. Software Engineer — 48.41%
2. Backend Developer — 11.34%
3. Python Developer — 5.63%
